# 02B - OCR seletivo

A extração nativa identificou PDFs sem camada textual. Esta etapa aplica OCR **somente** aos candidatos definidos em `config/experiment.yaml`.

Por padrão, o gatilho é `zero_text_only`, portanto os PDFs que já possuem texto não são submetidos a OCR.

In [1]:
import pandas as pd

from wonca_rag.config import load_experiment_config
from wonca_rag.paths import PATHS
from wonca_rag.ingestion.ocr import (
    available_languages,
    configure_tesseract,
    select_ocr_candidates,
    run_selective_ocr,
    save_ocr_report,
    summarize_ocr,
)

config = load_experiment_config()
EXTRACTION_REPORT = PATHS.outputs_root / 'logs' / 'pdf_extraction_report.parquet'
print('OCR config:', config.ocr.model_dump())

OCR config: {'enabled': True, 'engine': 'tesseract', 'trigger': 'zero_text_only', 'language': 'eng', 'dpi': 300, 'psm': 3, 'min_characters_after_ocr': 100}


## 1. Conferir Tesseract

In [2]:
tesseract_path = configure_tesseract()
langs = available_languages()
print('Tesseract:', tesseract_path)
print('Languages:', langs)

Tesseract: C:\Program Files\Tesseract-OCR\tesseract.exe
Languages: ['eng', 'osd', 'por']


## 2. Selecionar somente os candidatos

In [3]:
extraction_df = pd.read_parquet(EXTRACTION_REPORT)
candidates_df = select_ocr_candidates(
    extraction_report_df=extraction_df,
    trigger=config.ocr.trigger,
)

print('Candidatos a OCR:', len(candidates_df))
display(candidates_df[[
    'article_id', 'filename', 'relative_path',
    'n_pages', 'total_characters', 'likely_scanned'
]])

Candidatos a OCR: 11


,article_id,filename,relative_path,n_pages,total_characters,likely_scanned
0,ART_CC8D0D4F8DEE,Barker1989.pdf,Artigos sem resumo/Barker1989.pdf,5,0,True
1,ART_0080057343D5,Starfield1976b.pdf,Artigos sem resumo/Starfield1976b.pdf,2,0,True
2,ART_87286A208679,Starfield1993.pdf,Artigos sem resumo/Starfield1993.pdf,11,0,True
3,ART_00098517AA25,Bice1973.pdf,Bice1973.pdf,10,0,True
4,ART_4C1CCB6ED34C,Carr1993.pdf,Carr1993.pdf,5,0,True
5,ART_331895B7FCEB,Diaz1986.pdf,Diaz1986.pdf,9,0,True
6,ART_757497A5F654,Maguire1998.pdf,Maguire1998.pdf,10,0,True
7,ART_8AC4523EBB7F,Reid1996.pdf,Reid1996.pdf,7,0,True
8,ART_007D4934BC69,Riley1993.pdf,Riley1993.pdf,17,0,True
9,ART_34868B1241D9,Schor1987.pdf,Schor1987.pdf,11,0,True


## 3. Executar OCR seletivo

In [4]:
ocr_report_df = run_selective_ocr(
    candidates_df=candidates_df,
    pdf_root=PATHS.data_root / 'raw' / 'pdf',
    ocr_output_root=PATHS.data_root / 'processed' / 'ocr',
    checkpoints_root=PATHS.outputs_root / 'checkpoints' / 'ocr',
    language=config.ocr.language,
    dpi=config.ocr.dpi,
    psm=config.ocr.psm,
    min_characters_after_ocr=config.ocr.min_characters_after_ocr,
    resume=True,
)

display(ocr_report_df)

OCR PDFs:   0%|          | 0/11 [00:00<?, ?it/s]

,article_id,sha256,filename,relative_path,output_json,status,passes_min_characters,error_type,error_message,n_pages,total_characters,mean_characters_per_page,empty_pages,empty_page_ratio
0,ART_CC8D0D4F8DEE,cc8d0d4f8deed88fc7a21478e72b25555c65c120e35a8a...,Barker1989.pdf,Artigos sem resumo/Barker1989.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_CC8...,SKIPPED_CURRENT,True,None,None,5,15481,3096.200000,0,0.0
1,ART_0080057343D5,0080057343d5757eaf4f35c839a60a5145cfd9efc93680...,Starfield1976b.pdf,Artigos sem resumo/Starfield1976b.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_008...,SKIPPED_CURRENT,True,None,None,2,5709,2854.500000,0,0.0
2,ART_87286A208679,87286a208679738293580fe14f9d879d164be0146ba99a...,Starfield1993.pdf,Artigos sem resumo/Starfield1993.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_872...,SKIPPED_CURRENT,True,None,None,11,27764,2524.000000,0,0.0
3,ART_00098517AA25,00098517aa25cd3877b756c8cf0017765d00f4c4b503dd...,Bice1973.pdf,Bice1973.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_000...,SKIPPED_CURRENT,True,None,None,10,29729,2972.900000,0,0.0
4,ART_4C1CCB6ED34C,4c1ccb6ed34c3a94050fb4fa37ba0b8c9aa4982422aaae...,Carr1993.pdf,Carr1993.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_4C1...,SKIPPED_CURRENT,True,None,None,5,25877,5175.400000,0,0.0
5,ART_331895B7FCEB,331895b7fcebc11695ff71ab47c50d06ac0978ce3946c7...,Diaz1986.pdf,Diaz1986.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_331...,SKIPPED_CURRENT,True,None,None,9,30882,3431.333333,0,0.0
6,ART_757497A5F654,757497a5f6546ff11cd7098364f403e69ae32ac3d9da55...,Maguire1998.pdf,Maguire1998.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_757...,SKIPPED_CURRENT,True,None,None,10,41408,4140.800000,0,0.0
7,ART_8AC4523EBB7F,8ac4523ebb7fbb6e9950779f1f06006d508ebb23cd84f6...,Reid1996.pdf,Reid1996.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_8AC...,SKIPPED_CURRENT,True,None,None,7,39883,5697.571429,0,0.0
8,ART_007D4934BC69,007d4934bc695ab46fd3ce20a78ad6679c94b59cd042e0...,Riley1993.pdf,Riley1993.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_007...,SKIPPED_CURRENT,True,None,None,17,61508,3618.117647,0,0.0
9,ART_34868B1241D9,34868b1241d95cc134952f9bc039e6f1771fd989766957...,Schor1987.pdf,Schor1987.pdf,C:\Github\wonca-rag\data\processed\ocr\ART_348...,SKIPPED_CURRENT,True,None,None,11,39229,3566.272727,0,0.0


## 4. Resumo

In [5]:
summary = summarize_ocr(ocr_report_df)
display(pd.DataFrame(summary.items(), columns=['metric', 'value']))

,metric,value
0,n_candidates,11
1,n_done,0
2,n_skipped_current,11
3,n_error,0
4,n_pass_min_characters,11
5,n_still_zero_text,0


## 5. Conferir OCRs que ainda precisam de revisão

In [6]:
review_df = ocr_report_df.loc[
    (ocr_report_df['status'] == 'ERROR')
    | (~ocr_report_df['passes_min_characters'].fillna(False))
].copy()

print('Ainda precisam de revisão:', len(review_df))
display(review_df)

Ainda precisam de revisão: 0


,article_id,sha256,filename,relative_path,output_json,status,passes_min_characters,error_type,error_message,n_pages,total_characters,mean_characters_per_page,empty_pages,empty_page_ratio


## 6. Salvar relatório

In [7]:
paths = save_ocr_report(
    report_df=ocr_report_df,
    output_dir=PATHS.outputs_root / 'logs',
)
for name, path in paths.items():
    print(name, ':', path)

parquet : C:\Github\wonca-rag\outputs\logs\ocr_report.parquet
xlsx : C:\Github\wonca-rag\outputs\logs\ocr_report.xlsx
csv : C:\Github\wonca-rag\outputs\logs\ocr_report.csv


## Resultado

Os textos OCR ficam em `data/processed/ocr/`. A próxima etapa de chunking dará preferência ao JSON OCR quando ele existir, e usará a extração nativa para os demais artigos.